Rag pipeline-Data ingestion to vector DB pipeline

In [5]:
%pip install langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 5.6 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 2.3.0
    Uninstalling huggingface_hub-2.3.0:
      Successfully uninstalled huggingface_hub-2.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-huggingface]
Note: you may need to restart the kernel to use updated packages.


In [9]:
from pathlib import Path
from pypdf import PdfReader

# Your PDF folder
PDF_FOLDER = Path(
    "/Users/syedkhunmeer/Desktop/YTRAG/data/pdf"
)

# Check whether the folder exists
if not PDF_FOLDER.exists():
    raise FileNotFoundError(
        f"PDF folder does not exist: {PDF_FOLDER}"
    )

# Find PDF files
pdf_files = list(PDF_FOLDER.glob("*.pdf"))

print(f"PDF folder: {PDF_FOLDER}")
print(f"PDF files found: {len(pdf_files)}")

if not pdf_files:
    print("No PDFs found. Check the folder path and file extensions.")

documents = []

for pdf_file in pdf_files:
    try:
        reader = PdfReader(str(pdf_file))

        print(f"\nLoading: {pdf_file.name}")
        print(f"Pages: {len(reader.pages)}")

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""

            if text.strip():
                documents.append({
                    "source": str(pdf_file),
                    "page": page_number,
                    "text": text
                })

        print(f"Successfully processed: {pdf_file.name}")

    except Exception as error:
        print(f"FAILED: {pdf_file.name}")
        print(f"Reason: {type(error).__name__}: {error}")

print(f"\nTotal pages with extracted text: {len(documents)}")

if not documents:
    print(
        "\nNo text was extracted. Check the messages above. "
        "The PDFs may be scanned images, encrypted, or unreadable."
    )
else:
    print("\nPDF loading successful!")
    print("\nSample extracted text:")
    print(documents[0]["text"][:1000])

PDF folder: /Users/syedkhunmeer/Desktop/YTRAG/data/pdf
PDF files found: 1

Loading: sample_rag_research_paper.pdf
Pages: 3
Successfully processed: sample_rag_research_paper.pdf

Total pages with extracted text: 3

PDF loading successful!

Sample extracted text:
Sample RAG Research Paper    Page 1
 Retrieval-Augmented Generation for Question
 Answering
 A Sample Research-Style Document for Testing a PDF-Based RAG Pipeline
 Document type: Educational sample paper    Version: 1.0
Abstract
Retrieval-Augmented Generation (RAG) is a system design pattern that combines information retrieval with a
language model. Instead of relying only on information stored in a model's parameters, a RAG application
retrieves relevant passages from an external collection and provides those passages as context when generating
an answer. This sample document describes a typical RAG workflow, including document ingestion, text splitting,
embedding generation, vector search, context assembly, and answer generat

In [10]:
from pathlib import Path
import warnings

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


# ==========================================
# 1. CONFIGURATION
# ==========================================

PDF_DIRECTORY = Path(
    "/Users/syedkhunmeer/Desktop/YTRAG/data/pdf"
)

VECTOR_DB_DIRECTORY = Path(
    "/Users/syedkhunmeer/Desktop/YTRAG/vector_db"
)

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


# ==========================================
# 2. VALIDATE PDF DIRECTORY
# ==========================================

if not PDF_DIRECTORY.is_dir():
    raise FileNotFoundError(
        f"PDF directory does not exist: {PDF_DIRECTORY}"
    )

pdf_files = sorted(
    path for path in PDF_DIRECTORY.rglob("*")
    if path.is_file() and path.suffix.lower() == ".pdf"
)

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in: {PDF_DIRECTORY}"
    )

print(f"Found {len(pdf_files)} PDF file(s).")


# ==========================================
# 3. LOAD PDF DOCUMENTS
# ==========================================

documents = []

for pdf_file in pdf_files:
    try:
        print(f"Loading: {pdf_file.name}")

        loader = PyPDFLoader(str(pdf_file))
        pages = loader.load()

        for page in pages:
            page.metadata["source_file"] = pdf_file.name

        documents.extend(pages)

    except Exception as error:
        warnings.warn(
            f"Could not load {pdf_file.name}: {error}"
        )

if not documents:
    raise ValueError(
        "No PDF pages could be loaded successfully."
    )

print(f"Pages loaded: {len(documents)}")


# ==========================================
# 4. SPLIT DOCUMENTS INTO CHUNKS
# ==========================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True
)

chunks = [
    chunk
    for chunk in text_splitter.split_documents(documents)
    if chunk.page_content.strip()
]

if not chunks:
    raise ValueError(
        "No extractable text found. Scanned PDFs may require OCR."
    )

print(f"Chunks created: {len(chunks)}")


# ==========================================
# 5. INITIALIZE EMBEDDINGS
# ==========================================

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model initialized.")


# ==========================================
# 6. CREATE FAISS VECTOR DATABASE
# ==========================================

vector_db = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"Vectors indexed: {vector_db.index.ntotal}")


# ==========================================
# 7. SAVE VECTOR DATABASE
# ==========================================

VECTOR_DB_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

vector_db.save_local(str(VECTOR_DB_DIRECTORY))

print(f"Vector database saved: {VECTOR_DB_DIRECTORY}")


# ==========================================
# 8. TEST RETRIEVAL
# ==========================================

query = "What is the main topic of these documents?"

results = vector_db.similarity_search(
    query=query,
    k=min(4, vector_db.index.ntotal)
)

print("\n========== RETRIEVAL RESULTS ==========")

for number, document in enumerate(results, start=1):
    print(f"\nResult {number}")
    print(f"Source: {document.metadata.get('source_file', 'Unknown')}")
    print(f"Page: {document.metadata.get('page', 'Unknown')}")
    print(document.page_content)
    print("-" * 60)


# ==========================================
# 9. PIPELINE SUMMARY
# ==========================================

print("\n========== PIPELINE SUCCESSFUL ==========")
print(f"PDF files : {len(pdf_files)}")
print(f"Pages     : {len(documents)}")
print(f"Chunks    : {len(chunks)}")
print(f"Vectors   : {vector_db.index.ntotal}")
print(f"Saved to  : {VECTOR_DB_DIRECTORY}")

Found 1 PDF file(s).
Loading: sample_rag_research_paper.pdf
Pages loaded: 3
Chunks created: 9


/Users/syedkhunmeer/Desktop/YTRAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1027 14:54:44.076000 12353 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10298.05it/s]
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


Embedding model initialized.
Vectors indexed: 9
Vector database saved: /Users/syedkhunmeer/Desktop/YTRAG/vector_db

========== RETRIEVAL RESULTS ==========

Result 1
Source: sample_rag_research_paper.pdf
Page: 0
question, the system searches the collection for relevant passages and includes the strongest matches in the
model's prompt. The response can then be grounded in those passages.
2. Background and Core Concepts
Document ingestion. Ingestion extracts text and useful metadata from source files. For PDF files, the extraction
step should preserve page numbers and source filenames when possible. Empty, encrypted, or image-only PDFs
may require separate handling.
Chunking. Long documents are commonly divided into smaller text segments called chunks. Chunks that are too
large may contain unrelated information; chunks that are too small may lose context. Overlapping neighboring
chunks can preserve continuity across boundaries.
Embeddings. An embedding model maps text to a numerical vect